In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MP-DE").getOrCreate()

In [0]:
df = spark.read.table('cl_mp_de.`01_bronze`.bronze_inventory')
display(df)

### Standardizing column names

In [0]:
df_standarized = df.withColumnRenamed('%SKU_ID%', 'sku_id') \
    .withColumnRenamed('ST_ID_Ref', 'store_id') \
    .withColumnRenamed('^Stock-On-Hand^', 'stock_on_hand') \
    .withColumnRenamed('Last_Audit_Dt', 'last_audit_date')
display(df_standarized)

### Cleaning last_audit_date

In [0]:
from pyspark.sql.functions import to_date
#standardizing date format to yyyy-MM-dd
#casting last_audit_date to date
df_clean = df_standarized.withColumn('stock_on_hand', df_standarized['stock_on_hand'].cast('int')) \
    .withColumn('last_audit_date', 
        to_date('last_audit_date', 'yyyyMMdd').cast('date')
    )
display(df_clean)

In [0]:
df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cl_mp_de.`02_silver`.silver_inventory')